# Tool-calling Agent Arena with Eden AI

Same prompt, same four tools, four LLMs. Watch which models call the right tool, which hallucinate the answer instead of calling the tool, and which invent data when a tool returns *no results*.

Tool calling is the foundation of agents. The OpenAI-compatible `tools` parameter works identically across providers through Eden AI — only the `model` string changes. The interesting part is **how** each model uses (or fails to use) the tools.

**Prerequisites:** an Eden AI API key (set as `EDENAI_API_KEY` env var or in `.env`).

In [ ]:
%pip install --quiet aiohttp ipywidgets nest_asyncio python-dotenv requests

## 1. Configuration

Four models, all with `supports_function_calling: true` per the Eden AI catalog. Note the variance in capabilities — some support parallel tool calls, some don't:

| Model | parallel calls |
|---|---|
| Claude Sonnet 4.5 | ✗ |
| GPT-4o | ✓ |
| Gemini 2.5 Flash | ✓ |
| Mistral Large | ✗ |

In [ ]:
import base64
import json
import os

from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(override=True)

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY")
if not EDENAI_API_KEY:
    raise RuntimeError("Set EDENAI_API_KEY (env var or .env file). Get one at https://app.edenai.run")
EDENAI_URL = "https://api.edenai.run/v3/llm/chat/completions"

MODELS = [
    {"label": "Claude",  "model": "anthropic/claude-sonnet-4-5"},
    {"label": "GPT-4o",  "model": "openai/gpt-4o"},
    {"label": "Gemini",  "model": "google/gemini-2.5-flash"},
    {"label": "Mistral", "model": "mistral/mistral-large-latest"},
]

MAX_TURNS = 6  # safety cap on the agent loop


def _is_sandbox(jwt: str) -> bool:
    try:
        payload_b64 = jwt.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        return json.loads(base64.urlsafe_b64decode(payload_b64)).get("type") == "sandbox_api_token"
    except Exception:
        return False


if _is_sandbox(EDENAI_API_KEY):
    display(HTML(
        '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 14px;'
        'border-radius:4px;font-family:sans-serif;font-size:13px;margin:6px 0;">'
        '<b>⚠ Sandbox key detected.</b> Tool-calling responses on sandbox tokens may be '
        'mocked — the model might not actually decide to call tools. Use a production '
        'key for real agent behavior.</div>'
    ))

## 2. The tools

Four tools, all deterministic — no external API key needed, no network calls beyond the LLM itself. Two are *real* (calculator, get_current_time), two are *mocked with known-good + known-bad inputs* (get_weather, web_search). The mocks are the interesting part: they have a finite set of recognized inputs, so we can see what happens when a model asks about Atlantis (weather) or about Mars (search).

The trap pattern: **a tool returns "no data"**. The right behavior is for the model to respect the tool result. The wrong behavior is to invent an answer anyway.

In [ ]:
import math
from datetime import datetime
try:
    from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
except ImportError:
    from backports.zoneinfo import ZoneInfo, ZoneInfoNotFoundError

# Mocked data — finite known-good set + everything else returns no_data
WEATHER_MOCK = {
    "tokyo":    {"temp_c": 22, "condition": "cloudy"},
    "paris":    {"temp_c": 18, "condition": "rainy"},
    "new york": {"temp_c": 14, "condition": "clear"},
    "sydney":   {"temp_c": 26, "condition": "sunny"},
    "cairo":    {"temp_c": 31, "condition": "hot"},
}

SEARCH_MOCK = {
    "population of iceland":      "Iceland has approximately 380,000 inhabitants (2024 estimate).",
    "capital of australia":       "The capital of Australia is Canberra.",
    "speed of light":             "The speed of light in vacuum is 299,792,458 m/s.",
    "who wrote 1984":             "George Orwell wrote the novel 1984, published in 1949.",
}


def tool_calculator(expression: str) -> str:
    """Safely evaluate a math expression."""
    safe_names = {k: getattr(math, k) for k in ["sqrt", "pow", "sin", "cos", "tan", "log", "pi", "e"]}
    try:
        result = eval(expression, {"__builtins__": {}}, safe_names)
        return json.dumps({"result": result})
    except Exception as e:
        return json.dumps({"error": f"Could not evaluate: {e}"})


def tool_get_weather(city: str) -> str:
    data = WEATHER_MOCK.get(city.strip().lower())
    if data is None:
        return json.dumps({"city": city, "status": "no_data",
                           "message": "No weather data available for this city."})
    return json.dumps({"city": city, **data, "unit": "celsius"})


def tool_get_current_time(timezone: str) -> str:
    try:
        now = datetime.now(ZoneInfo(timezone))
        return json.dumps({
            "timezone": timezone,
            "iso": now.isoformat(),
            "hour": now.hour, "minute": now.minute,
        })
    except (ZoneInfoNotFoundError, Exception) as e:
        return json.dumps({"error": f"Unknown timezone {timezone!r}: {e}"})


def tool_web_search(query: str) -> str:
    answer = SEARCH_MOCK.get(query.strip().lower())
    if answer is None:
        return json.dumps({"query": query, "status": "no_results",
                           "message": "No search results found for this query."})
    return json.dumps({"query": query, "top_result": answer})


TOOL_FUNCTIONS = {
    "calculator":       tool_calculator,
    "get_weather":      tool_get_weather,
    "get_current_time": tool_get_current_time,
    "web_search":       tool_web_search,
}


TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate an arithmetic expression (supports +, -, *, /, **, sqrt, pi). Returns the numerical result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Arithmetic expression, e.g. '2 + 3 * sqrt(16)'."},
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Look up the current weather for a city. Returns temperature in Celsius and a condition string. Returns status='no_data' for unknown cities.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, e.g. 'Tokyo' or 'Paris'."},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current local time in a given IANA timezone (e.g. 'Europe/Paris', 'Asia/Tokyo', 'America/New_York').",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {"type": "string", "description": "IANA timezone identifier."},
                },
                "required": ["timezone"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for a query. Returns the top result. Returns status='no_results' if nothing was found.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query."},
                },
                "required": ["query"],
            },
        },
    },
]


def execute_tool(name: str, args: dict) -> str:
    fn = TOOL_FUNCTIONS.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        return fn(**args)
    except TypeError as e:
        return json.dumps({"error": f"Bad arguments to {name}: {e}"})

## 3. The agent loop

OpenAI-compatible tool calling is a two-or-more-turn protocol:

1. Send the user's message + `tools` schema
2. Model responds with either `content` (final answer) or `tool_calls` (one or more)
3. If tool_calls: execute each tool, append the assistant message + tool result messages
4. Repeat until the model returns content (no more tool_calls) or we hit `MAX_TURNS`

We track every step so we can render the full transcript per model — and see which models call too few tools (and hallucinate) vs too many tools (and waste turns).

In [ ]:
import asyncio
import time

import aiohttp

MAX_RETRIES = 2


async def _call_llm(session, payload):
    headers = {
        "Authorization": f"Bearer {EDENAI_API_KEY}",
        "Content-Type": "application/json",
    }
    for attempt in range(MAX_RETRIES + 1):
        async with session.post(EDENAI_URL, headers=headers, json=payload,
                                timeout=aiohttp.ClientTimeout(total=90)) as resp:
            body = await resp.text()
            if resp.status == 200:
                return json.loads(body)
            if resp.status in (400, 429, 502, 503, 504) and attempt < MAX_RETRIES:
                await asyncio.sleep(0.6 * (attempt + 1))
                continue
            raise RuntimeError(f"HTTP {resp.status}: {body[:200]}")
    raise RuntimeError("exhausted retries")


def _clean_tool_call(tc):
    """Strip provider-specific noise from tool_calls before sending them back
    in the next turn (some providers reject unknown fields)."""
    return {
        "id":   tc["id"],
        "type": tc.get("type", "function"),
        "function": {
            "name":      tc["function"]["name"],
            "arguments": tc["function"]["arguments"],
        },
    }


async def run_agent(session, model_cfg, user_prompt):
    """Run the multi-turn agent loop. Returns a dict with the full trace."""
    messages = [{"role": "user", "content": user_prompt}]
    trace = []  # list of {"kind": "tool_call"|"tool_result"|"final"|"error", ...}
    start = time.perf_counter()
    tool_call_count = 0
    turn = 0

    try:
        while turn < MAX_TURNS:
            turn += 1
            payload = {"model": model_cfg["model"], "messages": messages, "tools": TOOLS_SCHEMA}
            data = await _call_llm(session, payload)
            msg = data["choices"][0]["message"]

            tool_calls = msg.get("tool_calls") or []

            if not tool_calls:
                trace.append({"kind": "final", "text": msg.get("content") or ""})
                return {
                    "label": model_cfg["label"], "status": "ok",
                    "latency": time.perf_counter() - start,
                    "turns": turn, "tool_calls": tool_call_count,
                    "final": msg.get("content") or "",
                    "trace": trace,
                }

            # Append the assistant message (with cleaned tool_calls) for the next turn.
            messages.append({
                "role": "assistant",
                "content": msg.get("content"),
                "tool_calls": [_clean_tool_call(tc) for tc in tool_calls],
            })

            for tc in tool_calls:
                tool_call_count += 1
                name = tc["function"]["name"]
                try:
                    args = json.loads(tc["function"]["arguments"] or "{}")
                except json.JSONDecodeError:
                    args = {}
                result = execute_tool(name, args)
                trace.append({"kind": "tool_call", "name": name, "args": args})
                trace.append({"kind": "tool_result", "name": name, "result": result})
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc["id"],
                    "content": result,
                })

        # Hit the MAX_TURNS cap without a final answer
        return {
            "label": model_cfg["label"], "status": "max_turns",
            "latency": time.perf_counter() - start,
            "turns": turn, "tool_calls": tool_call_count,
            "final": "", "trace": trace,
        }

    except Exception as e:
        trace.append({"kind": "error", "text": str(e)})
        return {
            "label": model_cfg["label"], "status": "error",
            "latency": time.perf_counter() - start,
            "turns": turn, "tool_calls": tool_call_count,
            "final": "", "trace": trace, "error": str(e),
        }

## 4. UI

Type a prompt (or click one of the canned sample prompts) and press **⚔ Fight**. Each panel shows the model's full trace: 🔧 tool calls with arguments → ← tool results → 💬 final answer. The Compare table aggregates tool usage and final answers side-by-side.

In [ ]:
from ipywidgets import (
    Button, GridBox, HBox, HTML as HTMLWidget,
    Layout, Output, Textarea, ToggleButtons, VBox,
)
from IPython.display import display

SAMPLE_PROMPTS = [
    ("💰 Calc",       "What is 17% of $84.50?"),
    ("🌡 Weather + convert", "What's the weather in Tokyo, and convert that temperature to Fahrenheit?"),
    ("🪤 Unknown city (trap)", "What's the weather in Atlantis?"),
    ("🕒 Timezone",   "I'm in Paris. My meeting is at 9 AM Tokyo time. What time is that for me?"),
    ("🔎 Search trap", "What's the latest scientific news about Mars?"),
    ("🧠 No tool needed", "What is 2 + 2?"),
]

prompt_box = Textarea(
    value=SAMPLE_PROMPTS[0][1],
    placeholder="Ask anything — let the models decide which tools to call…",
    layout=Layout(width="100%", height="60px"),
)

sample_btns = []
for label, text in SAMPLE_PROMPTS:
    b = Button(description=label, layout=Layout(width="170px"))
    def _make_setter(t):
        return lambda _: setattr(prompt_box, "value", t)
    b.on_click(_make_setter(text))
    sample_btns.append(b)
sample_row1 = HBox(sample_btns[:3])
sample_row2 = HBox(sample_btns[3:])

fight_btn = Button(description="⚔  Fight", button_style="primary")
clear_btn = Button(description="Clear")
view_toggle = ToggleButtons(
    options=[("Trace panels", "panels"), ("Compare table", "table")],
    value="panels",
    style={"button_width": "140px"},
)

panel_layout = Layout(border="1px solid #ddd", padding="8px", height="340px", overflow="auto")
panels = [Output(layout=panel_layout) for _ in MODELS]
headers = [HTMLWidget() for _ in MODELS]


def _empty_header(i):
    m = MODELS[i]
    return (
        f'<div style="font-family:sans-serif;font-size:13px;padding:4px;">'
        f'<b>{m["label"]}</b> <span style="color:#888;font-size:11px;">{m["model"]}</span></div>'
    )


def _set_empty_panel(i):
    headers[i].value = _empty_header(i)
    panels[i].clear_output()
    with panels[i]:
        display(HTML(
            '<div style="font-family:sans-serif;color:#aaa;font-size:12px;text-align:center;padding:50px 10px;">'
            'Pick a prompt and click<br><b>⚔ Fight</b><br>to send to all 4 models</div>'
        ))


for i in range(len(MODELS)):
    _set_empty_panel(i)

panel_blocks = [
    VBox([headers[i], panels[i]], layout=Layout(border="1px solid #eee", padding="4px", border_radius="4px"))
    for i in range(len(MODELS))
]
grid = GridBox(
    panel_blocks,
    layout=Layout(grid_template_columns="repeat(2, 1fr)", grid_gap="8px"),
)

table_view = Output(layout=Layout(border="1px solid #eee", padding="8px", border_radius="4px", display="none"))


def _on_view_change(change):
    if change["new"] == "panels":
        grid.layout.display = ""
        table_view.layout.display = "none"
    else:
        grid.layout.display = "none"
        table_view.layout.display = ""


view_toggle.observe(_on_view_change, names="value")

summary_out = Output()

display(VBox([
    HTMLWidget(value='<b style="font-family:sans-serif;font-size:13px;">Prompt</b>'),
    prompt_box,
    HTMLWidget(value='<span style="font-family:sans-serif;font-size:11px;color:#666;">Sample prompts:</span>'),
    sample_row1,
    sample_row2,
    HBox([fight_btn, clear_btn, view_toggle]),
    grid,
    table_view,
    summary_out,
]))

## 5. Wire it up

Fight → fan out to all 4 models in parallel via `asyncio.gather`. Each model runs its own agent loop independently.

In [ ]:
import html as _html

import nest_asyncio
from IPython.display import clear_output

nest_asyncio.apply()

display(HTML('''
<style>
@keyframes cb_blink { 0%, 100% { opacity: 0.2; } 50% { opacity: 1; } }
.cb-dot { animation: cb_blink 1.2s infinite both; display:inline-block; }
.cb-dot:nth-child(2) { animation-delay: 0.2s; }
.cb-dot:nth-child(3) { animation-delay: 0.4s; }
</style>
'''))

STATUS_STYLES = {
    "ok":        ("done ✓",      "#28a745"),
    "max_turns": ("max turns ⚠", "#fd7e14"),
    "error":     ("error ✗",     "#dc3545"),
}


def _render_header(i, status_key, latency, tool_calls, turns):
    m = MODELS[i]
    status_label, color = STATUS_STYLES.get(status_key, (status_key, "#6c757d"))
    badge = (
        f'<span style="background:{color};color:white;padding:3px 10px;border-radius:10px;'
        f'font-size:11px;font-weight:600;">{status_label}</span> '
        f'<span style="color:#666;font-size:11px;">'
        f'{tool_calls} tool call{"s" if tool_calls != 1 else ""} · {turns} turn{"s" if turns != 1 else ""} · {latency:.2f}s</span>'
    )
    headers[i].value = (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px;gap:8px;flex-wrap:wrap;">'
        f'  <div><b style="font-size:13px;">{m["label"]}</b> '
        f'<span style="color:#888;font-size:11px;">{m["model"]}</span></div>'
        f'  <div>{badge}</div>'
        '</div>'
    )


def _set_loading_header(i):
    m = MODELS[i]
    headers[i].value = (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px;">'
        f'  <div><b style="font-size:13px;">{m["label"]}</b> '
        f'<span style="color:#888;font-size:11px;">{m["model"]}</span></div>'
        '  <div><span style="background:#17a2b8;color:white;padding:3px 10px;'
        'border-radius:10px;font-size:11px;font-weight:600;">running'
        '<span class="cb-dot">.</span><span class="cb-dot">.</span><span class="cb-dot">.</span></span></div>'
        '</div>'
    )


def _render_trace_html(trace):
    blocks = []
    for step in trace:
        if step["kind"] == "tool_call":
            args_str = _html.escape(json.dumps(step["args"], ensure_ascii=False))
            blocks.append(
                f'<div style="font-family:monospace;font-size:11px;color:#0066cc;'
                f'background:#e7f3ff;padding:4px 8px;border-radius:3px;margin:3px 0;">'
                f'🔧 <b>{step["name"]}</b>({args_str})</div>'
            )
        elif step["kind"] == "tool_result":
            try:
                parsed = json.loads(step["result"])
                result_str = json.dumps(parsed, ensure_ascii=False)
            except Exception:
                result_str = step["result"]
            result_str = _html.escape(result_str)
            # Detect "no data" / "no results" / error replies to color the trap warnings
            is_empty = ("no_data" in step["result"] or
                        "no_results" in step["result"] or
                        '"error"' in step["result"])
            bg = "#fff3cd" if is_empty else "#f0fdf4"
            color = "#92400e" if is_empty else "#166534"
            blocks.append(
                f'<div style="font-family:monospace;font-size:11px;color:{color};'
                f'background:{bg};padding:4px 8px;border-radius:3px;margin:3px 0 6px 16px;">'
                f'← {result_str}</div>'
            )
        elif step["kind"] == "final":
            text = _html.escape(step["text"]).replace("\n", "<br>")
            blocks.append(
                f'<div style="font-family:sans-serif;font-size:12px;background:#f8f9fa;'
                f'padding:8px 10px;border-radius:4px;margin-top:8px;border-left:3px solid #007bff;">'
                f'💬 {text}</div>'
            )
        elif step["kind"] == "error":
            blocks.append(
                f'<div style="font-family:monospace;font-size:11px;color:#dc3545;'
                f'background:#f8d7da;padding:6px 10px;border-radius:3px;margin:6px 0;">'
                f'✗ {_html.escape(step["text"])}</div>'
            )
    return "".join(blocks) or '<div style="color:#888;font-family:sans-serif;font-size:12px;">(no trace)</div>'


def _render_panel(i, result):
    _render_header(i, result["status"], result["latency"],
                   result["tool_calls"], result["turns"])
    panels[i].clear_output()
    with panels[i]:
        display(HTML(_render_trace_html(result["trace"])))


def _render_table(results):
    # Aggregate per-model: tools used (set), tool-call count, final-answer summary
    rows = ""
    head = (
        '<thead><tr>'
        '<th style="text-align:left;padding:6px 10px;background:#f8f9fa;border-bottom:2px solid #ddd;">Model</th>'
        '<th style="text-align:left;padding:6px 10px;background:#f8f9fa;border-bottom:2px solid #ddd;">Tools called</th>'
        '<th style="text-align:left;padding:6px 10px;background:#f8f9fa;border-bottom:2px solid #ddd;"># calls</th>'
        '<th style="text-align:left;padding:6px 10px;background:#f8f9fa;border-bottom:2px solid #ddd;">Turns</th>'
        '<th style="text-align:left;padding:6px 10px;background:#f8f9fa;border-bottom:2px solid #ddd;">Latency</th>'
        '<th style="text-align:left;padding:6px 10px;background:#f8f9fa;border-bottom:2px solid #ddd;">Final answer</th>'
        '</tr></thead>'
    )
    for r in results:
        tools_used = sorted({step["name"] for step in r["trace"] if step["kind"] == "tool_call"})
        tools_label = ", ".join(tools_used) if tools_used else '<span style="color:#dc3545;">none</span>'
        final = _html.escape(r["final"])[:300]
        status_label, color = STATUS_STYLES.get(r["status"], (r["status"], "#6c757d"))
        rows += (
            '<tr>'
            f'<td style="padding:6px 10px;border-bottom:1px solid #eee;font-family:sans-serif;font-size:12px;"><b>{r["label"]}</b></td>'
            f'<td style="padding:6px 10px;border-bottom:1px solid #eee;font-family:monospace;font-size:11px;">{tools_label}</td>'
            f'<td style="padding:6px 10px;border-bottom:1px solid #eee;font-family:monospace;font-size:11px;">{r["tool_calls"]}</td>'
            f'<td style="padding:6px 10px;border-bottom:1px solid #eee;font-family:monospace;font-size:11px;">{r["turns"]}</td>'
            f'<td style="padding:6px 10px;border-bottom:1px solid #eee;font-family:monospace;font-size:11px;">{r["latency"]:.2f}s</td>'
            f'<td style="padding:6px 10px;border-bottom:1px solid #eee;font-family:sans-serif;font-size:11px;max-width:380px;">{final}</td>'
            '</tr>'
        )
    with table_view:
        clear_output()
        display(HTML(
            '<table style="border-collapse:collapse;width:100%;">'
            + head + '<tbody>' + rows + '</tbody></table>'
        ))


def _render_summary(results):
    fastest = min(results, key=lambda r: r["latency"])
    most_tools = max(results, key=lambda r: r["tool_calls"])
    no_tool = [r["label"] for r in results if r["tool_calls"] == 0 and r["status"] == "ok"]
    failed = [r["label"] for r in results if r["status"] != "ok"]
    parts = [
        f'<span style="color:#666;">⚡ Fastest: <b>{fastest["label"]}</b> ({fastest["latency"]:.2f}s)</span>',
        f'<span style="color:#666;">🔧 Most tool calls: <b>{most_tools["label"]}</b> ({most_tools["tool_calls"]})</span>',
    ]
    if no_tool:
        parts.append(f'<span style="color:#dc3545;">🤔 Answered without tools: {", ".join(no_tool)}</span>')
    if failed:
        parts.append(f'<span style="color:#dc3545;">✗ Failed: {", ".join(failed)}</span>')
    with summary_out:
        clear_output()
        display(HTML(
            f'<div style="background:#f8f9fa;padding:10px 12px;border-radius:4px;'
            f'border-left:4px solid #007bff;font-family:sans-serif;font-size:13px;'
            f'display:flex;gap:18px;flex-wrap:wrap;">{"".join(parts)}</div>'
        ))


async def run_round(user_prompt):
    for i in range(len(MODELS)):
        panels[i].clear_output()
        with panels[i]:
            display(HTML(
                '<div style="font-family:sans-serif;color:#17a2b8;font-size:13px;text-align:center;padding:50px 10px;">'
                'running<span class="cb-dot">.</span><span class="cb-dot">.</span><span class="cb-dot">.</span></div>'
            ))
        _set_loading_header(i)
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(*[
            run_agent(session, MODELS[i], user_prompt)
            for i in range(len(MODELS))
        ])
    for i, r in enumerate(results):
        _render_panel(i, r)
    _render_table(results)
    _render_summary(results)
    return results


def on_fight(_):
    p = prompt_box.value.strip()
    if not p:
        return
    asyncio.run(run_round(p))


def on_clear(_):
    for i in range(len(MODELS)):
        _set_empty_panel(i)
    table_view.clear_output()
    summary_out.clear_output()


fight_btn.on_click(on_fight)
clear_btn.on_click(on_clear)

## 6. The failure modes worth watching

Each sample prompt was chosen to surface a specific failure pattern:

- **💰 Calc** — pure arithmetic. Every model should call `calculator`. A model that *doesn't* (and computes 17% × $84.50 mentally) often produces small rounding errors.
- **🌡 Weather + convert** — chained tools. Needs `get_weather` then `calculator`. Reveals chain-of-tools planning quality.
- **🪤 Unknown city (trap)** — the killer demo. `get_weather('Atlantis')` returns `status: "no_data"`. **Does the model respect that, or hallucinate weather for Atlantis anyway?** A genuinely useful agent says "no data available". A bad one says "sunny, 22°C".
- **🕒 Timezone** — needs `get_current_time(timezone)` twice (Paris + Tokyo) plus reasoning. Tests parallel-vs-sequential tool calling.
- **🔎 Search trap** — `web_search('Mars news')` returns `no_results`. Same trap as the unknown-city one, applied to search. Watch for invented news headlines.
- **🧠 No tool needed** — the *over-eager* failure mode. The right answer to "What is 2+2?" is 4, with no `calculator` call. A model that calls `calculator("2+2")` is wasting a turn (and proves it can't tell trivial from non-trivial).

## 7. Customize

**Add a tool.** Define a Python function, register it, and add its JSON schema:

```python
def tool_send_email(to: str, subject: str, body: str) -> str:
    # In production, wire to your email backend.
    return json.dumps({"status": "sent", "to": to})

TOOL_FUNCTIONS["send_email"] = tool_send_email
TOOLS_SCHEMA.append({
    "type": "function",
    "function": {
        "name": "send_email",
        "description": "Send an email. Returns status='sent' on success.",
        "parameters": {
            "type": "object",
            "properties": {
                "to":      {"type": "string"},
                "subject": {"type": "string"},
                "body":    {"type": "string"},
            },
            "required": ["to", "subject", "body"],
        },
    },
})
```

**Constrain tool choice.** Force a specific tool with `tool_choice` (all 4 models support it):

```python
payload["tool_choice"] = {"type": "function", "function": {"name": "calculator"}}
```

Useful for building a *deterministic* agent that always uses one specific tool.

**Swap the model lineup.** Any model with `supports_function_calling: true` works. Notable picks:
- `anthropic/claude-haiku-4-5` — cheapest with tool calling
- `deepseek/deepseek-chat` — open-weight, supports parallel calls
- `openai/gpt-4.1` — newer than gpt-4o, supports parallel calls

---

Tried it? Open an issue with a prompt + the model that surprised you most — especially the trap prompts.